# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/syeddaniyalg/flyrank-work/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

Looking at `impressions_90d`, `ctr`, and `avg_position` before testing anything.
Traffic metrics are almost always heavy-tailed, a few pages carry most of the volume,
most pages sit in a long thin tail. Log-transform impressions before any correlation work.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
import os

os.chdir("./../../")
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("impressions_90d describe:")
print(df["impressions_90d"].describe())
print(f"skew: {df['impressions_90d'].skew():.2f}")

print("\nctr describe:")
print(df["ctr"].describe())

print("\navg_position describe (0 means no data):")
print(df.loc[df["avg_position"] > 0, "avg_position"].describe())

df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
print("\nlog1p impressions_90d skew after transform:")
print(f"{df['log_impressions_90d'].skew():.2f}")

impressions_90d describe:
count     30000.000000
mean       5200.366300
std       16838.019547
min           1.000000
25%          81.000000
50%         731.000000
75%        3615.250000
max      517715.000000
Name: impressions_90d, dtype: float64
skew: 11.38

ctr describe:
count    30000.000000
mean         0.510733
std          3.279162
min          0.000000
25%          0.000000
50%          0.070000
75%          0.290000
max        100.000000
Name: ctr, dtype: float64

avg_position describe (0 means no data):
count    28795.000000
mean        17.026268
std         15.152439
min          0.100000
25%          6.700000
50%         11.400000
75%         22.900000
max        245.000000
Name: avg_position, dtype: float64

log1p impressions_90d skew after transform:
-0.39


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
visible = df[df["impressions_90d"] >= 100].copy()

tier_order = ["top_3", "page_1", "striking", "page_3_5", "deep", "no_data"]
test1 = (visible.groupby("position_tier")["ctr"]
         .agg(["mean", "count"]).reindex(tier_order).dropna())
print("Test 1 claim: CTR is higher at better position tiers")
print(test1)
v1 = "CONFIRMED" if test1["mean"].is_monotonic_decreasing else "MIXED"
print(f"Verdict: {v1}\n")

age_bins = [0, 90, 365, 730, np.inf]
age_labels = ["under_90d", "90_to_365d", "365_to_730d", "over_730d"]
visible["age_bucket"] = pd.cut(visible["content_age_days"], bins=age_bins, labels=age_labels)
test2 = visible.groupby("age_bucket", observed=True)["ctr"].agg(["mean", "count"])
print("Test 2 claim: older content has lower CTR")
print(test2)
v2 = "CONFIRMED" if test2["mean"].is_monotonic_decreasing else "MIXED"
print(f"Verdict: {v2}\n")

wc_bins = [0, 500, 1500, 3000, np.inf]
wc_labels = ["under_500", "500_to_1500", "1500_to_3000", "over_3000"]
visible["wc_bucket"] = pd.cut(visible["word_count"], bins=wc_bins, labels=wc_labels)
test3 = visible.groupby("wc_bucket", observed=True)["sessions_90d"].agg(["mean", "count"])
print("Test 3 claim: longer pages get more traffic")
print(test3)
v3 = "CONFIRMED" if test3["mean"].is_monotonic_increasing else "MIXED"
print(f"Verdict: {v3}")

Test 1 claim: CTR is higher at better position tiers
                   mean   count
position_tier                  
top_3          0.334128   533.0
page_1         0.354760  8633.0
striking       0.255782  5903.0
page_3_5       0.142359  6058.0
deep           0.055415   879.0
Verdict: MIXED

Test 2 claim: older content has lower CTR
                 mean  count
age_bucket                  
under_90d    0.466941    304
90_to_365d   0.274999  16367
365_to_730d  0.190978   5335
Verdict: CONFIRMED

Test 3 claim: longer pages get more traffic
                   mean  count
wc_bucket                     
500_to_1500   17.933440   1247
1500_to_3000  31.286469   7191
over_3000     87.475260   7013
Verdict: CONFIRMED


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
vol_bins = [0, 100, 500, 2000, np.inf]
vol_labels = ["under_100", "100_to_500", "500_to_2000", "over_2000"]
df["volume_bucket"] = pd.cut(df["impressions_90d"], bins=vol_bins, labels=vol_labels)

flag_test = df.groupby("volume_bucket", observed=True)["ctr"].agg(["std", "mean", "count"])
print("Flag-linked claim, behind quick-win logic: CTR is only trustworthy above a volume floor")
print(flag_test)

falls = flag_test["std"].is_monotonic_decreasing
verdict = "CONFIRMED" if falls else "MIXED"
print(f"Verdict: {verdict}")
print("CTR volatility drops as volume rises, supports gating any CTR-based rule behind a minimum "
      "impression floor, exactly what the quick-win logic assumes" if falls
      else "Volatility does not fall cleanly, the floor choice needs revisiting before trusting quick-win flags")

Flag-linked claim, behind quick-win logic: CTR is only trustworthy above a volume floor
                    std      mean  count
volume_bucket                           
under_100      6.260319  1.207503   8006
100_to_500     0.594428  0.240547   5279
500_to_2000    0.300344  0.206372   6502
over_2000      0.322084  0.297958  10213
Verdict: MIXED
Volatility does not fall cleanly, the floor choice needs revisiting before trusting quick-win flags


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("What a content team should take from this:")
print("1. CTR comparisons only make sense within a position tier, not across the whole site,")
print("   since tier alone explains most of the CTR spread.")
print("2. Any CTR-based flag needs a volume floor (this data supports ~100 impressions/90d),")
print("   below that floor a single click swings the rate too much to trust.")
print("3. Content age on its own is a weaker lever than tier or volume for this lane,")
print("   worth checking again next quarter but not worth a standalone rule yet.")

What a content team should take from this:
1. CTR comparisons only make sense within a position tier, not across the whole site,
   since tier alone explains most of the CTR spread.
2. Any CTR-based flag needs a volume floor (this data supports ~100 impressions/90d),
   below that floor a single click swings the rate too much to trust.
3. Content age on its own is a weaker lever than tier or volume for this lane,
   worth checking again next quarter but not worth a standalone rule yet.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.